In [97]:
# Import necessary libraries and dependencies
import pandas as pd
from datetime import datetime, timedelta

In [98]:
# Load the dataset to a pandas DataFrame
zulo_bank = pd.read_csv('dataset/zulo_bank.csv')

In [ ]:
# Display the first 5 rows of the dataset
display(zulo_bank.head(5))

In [ ]:
# Check the data set
zulo_bank.info()

In [ ]:
# Fill up the missing values with the appropriate parameters
zulo_bank.fillna(
    {
        "LoanAmount": 0.0,
        "LoanType": "Unknown",
        "InterestRate": 0.0,
    },
    inplace=True,
)

In [ ]:
# Check the data set again after filling missing values
zulo_bank.info()

In [100]:
# Convert to 1NF: First Normal Form
# Spplitting the 'FullName' column into 'first_name' and 'last_name'
zulo_bank[["first_name", "last_name"]] = zulo_bank["FullName"].str.split(expand=True)

In [101]:
# Converting 1NF to 2NF: Second Normal Form
# Customers table
customer = (
    zulo_bank[["first_name", "last_name", "Email", "Phone"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
customer["customer_id"] = range(1, len(customer) + 1)
customer = customer[["customer_id", "first_name", "last_name", "Email", "Phone"]]

In [102]:
# Create the Accounts table
account = (
    zulo_bank[["AccountType", "Balance", "OpeningDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
account["account_id"] = range(1, len(account) + 1)
account = account[["account_id", "AccountType", "Balance", "OpeningDate"]]

In [103]:
# Create transactions table
transaction = (
    zulo_bank[["TransactionType", "Amount", "TransactionDate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
transaction["transaction_id"] = range(1, len(transaction) + 1)
transaction = transaction[
    ["transaction_id", "TransactionType", "Amount", "TransactionDate"]
]

In [104]:
# Create a Loans table
loan = (
    zulo_bank[["LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)
loan["loan_id"] = range(1, len(loan) + 1)
loan = loan[
    ["loan_id", "LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"]
]

In [105]:
# Merge operations to create the zulo_bank
zulo_bank = (
    zulo_bank.merge(
        customer, on=["first_name", "last_name", "Email", "Phone"], how="left"
    )
    .merge(account, on=["AccountType", "Balance", "OpeningDate"], how="left")
    .merge(transaction, on=["TransactionType", "Amount", "TransactionDate"], how="left")
    .merge(
        loan,
        on=["LoanAmount", "LoanType", "StartDate", "EndDate", "InterestRate"],
        how="left",
    )[["customer_id", "account_id", "transaction_id", "loan_id"]]
)

In [ ]:
zulo_bank

In [106]:
# Convert 2NF to 3NF: Third Normal Form
# Create the date dimension table
# Define the start and end dates for the date dimension
start_date = datetime(2020, 1, 1)
current_date = datetime(2090, 12, 31)

# Calculate the number of days between start date and current date
num_days = (current_date - start_date).days

# Generate a list of dates from start date to current date
date_list = [start_date + timedelta(days=x) for x in range(num_days + 1)]

# Ensure date_id matches the length of date_list
date = {"date_id": [x for x in range(1, len(date_list) + 1)], "date": date_list}

# Create DataFrame
date_dim = pd.DataFrame(date)
date_dim["Year"] = date_dim["date"].dt.year
date_dim["Month"] = date_dim["date"].dt.month
date_dim["Day"] = date_dim["date"].dt.day
date_dim["date"] = pd.to_datetime(date_dim["date"]).dt.date

In [ ]:
date_dim

In [107]:
# Account table 2NF to 3NF
account["OpeningDate"] = pd.to_datetime(account["OpeningDate"]).dt.date
account = (
    account.merge(date_dim, left_on="OpeningDate", right_on="date", how="inner")
    .rename(columns={"date_id": "OpeningDate_ID"})
    .reset_index(drop=True)[["account_id", "AccountType", "Balance", "OpeningDate_ID"]]
)

In [108]:
# Create the transaction table 2NF to 3NF
transaction["TransactionDate"] = pd.to_datetime(transaction["TransactionDate"]).dt.date
transaction = (
    transaction.merge(date_dim, left_on="TransactionDate", right_on="date", how="inner")
    .rename(columns={"date_id": "TransactionDate_ID"})
    .reset_index(drop=True)[
        ["transaction_id", "TransactionType", "Amount", "TransactionDate_ID"]
    ]
)   

In [ ]:
transaction

In [109]:
# Create Loan table 2NF to 3NF
loan["StartDate"] = pd.to_datetime(loan["StartDate"]).dt.date 
loan["EndDate"] = pd.to_datetime(loan["EndDate"]).dt.date
loan = (
    loan.merge(date_dim, left_on="StartDate", right_on="date", how="inner")
    .rename(columns={"date_id": "StartDate_ID"})
    .merge(
        date_dim, left_on="EndDate", right_on="date", how="inner", suffixes=("", "_end")
    )
    .rename(columns={"date_id": "EndDate_ID"})[
        [
            "loan_id",
            "LoanAmount",
            "LoanType",
            "StartDate_ID",
            "EndDate_ID",
            "InterestRate",
        ]
    ]
)

In [ ]:
loan